In [14]:
!pip install transformers datasets torch scikit-learn pandas anthropic -q

In [17]:
# Minor Safety Content Detection - Open Source Project
# Implementation references HuggingFace docs and related tutorials

# Optional: Download Jigsaw dataset from Kaggle (upload kaggle.json first in Colab)
# from google.colab import files; files.upload()
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c jigsaw-toxic-comment-classification-challenge
# !unzip jigsaw-toxic-comment-classification-challenge.zip -d jigsaw/

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
)
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
import warnings
warnings.filterwarnings("ignore")

# Part 1: Data Loading
# Using Jigsaw Toxic Comment dataset - has multi-label annotations we convert to binary
def load_data(path="train.csv", sample_size=20000):
    # Load CSV, create binary label: 1 if any toxic category present else 0
    # Sampling to keep training time reasonable (full dataset would take too long)
    df = pd.read_csv(path)
    toxic_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
    df["label"] = (df[toxic_cols].sum(axis=1) > 0).astype(int)
    df = df[["comment_text", "label"]].dropna()

    # Balanced sampling: equal toxic/non-toxic to avoid class imbalance affecting training
    toxic = df[df.label == 1].sample(min(sample_size // 2, df.label.sum()), random_state=42)
    clean = df[df.label == 0].sample(sample_size // 2, random_state=42)
    df = pd.concat([toxic, clean]).sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"Dataset size: {len(df)} | Toxic: {df.label.sum()} | Clean: {(df.label==0).sum()}")
    return df

df = load_data()
split = int(len(df) * 0.8)
train_df = df[:split].reset_index(drop=True)
test_df  = df[split:].reset_index(drop=True)

# Part 2: Tokenization and Dataset
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class ToxicDataset(Dataset):
    # Convert comments to tokenized format for the model (max_len=128 keeps sequences manageable)
    def __init__(self, df, tokenizer, max_len=128):
        self.encodings = tokenizer(
            list(df.comment_text),
            truncation=True, padding=True,
            max_length=max_len
        )
        self.labels = list(df.label)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

train_dataset = ToxicDataset(train_df, tokenizer)
test_dataset  = ToxicDataset(test_df,  tokenizer)

# Part 3: Model Fine-tuning
# DistilBERT chosen for good balance between accuracy and speed (lighter than full BERT)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    # Compute F1, precision, recall - F1 is most informative for imbalanced-ish binary classification
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "f1":        f1_score(labels, preds),
        "precision": precision_score(labels, preds),
        "recall":    recall_score(labels, preds),
    }

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=True,  # Mixed precision for faster training when GPU available
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()
print("Training complete.")

# Part 4: Evaluation and Error Analysis
# Analyze where the model fails - helps understand limitations and potential improvements
def evaluate_and_analyze(trainer, test_df, test_dataset, top_n=20):
    preds_output = trainer.predict(test_dataset)
    preds  = np.argmax(preds_output.predictions, axis=1)
    labels = preds_output.label_ids

    print("\n" + "="*60)
    print("OVERALL PERFORMANCE")
    print("="*60)
    print(classification_report(labels, preds, target_names=["Clean", "Toxic"]))

    cm = confusion_matrix(labels, preds)
    print(f"Confusion Matrix:\n{cm}")
    tn, fp, fn, tp = cm.ravel()
    print(f"True Negatives: {tn} | False Positives: {fp}")
    print(f"False Negatives: {fn} | True Positives: {tp}")

    # Extract misclassified samples for manual inspection
    test_df = test_df.copy()
    test_df["pred"]  = preds
    test_df["label"] = labels
    test_df["text_len"] = test_df.comment_text.str.len()

    false_positives = test_df[(test_df.pred == 1) & (test_df.label == 0)]
    false_negatives = test_df[(test_df.pred == 0) & (test_df.label == 1)]

    print("\n" + "="*60)
    print(f"ERROR ANALYSIS: {len(false_positives)} False Positives (Clean → flagged as Toxic)")
    print("="*60)
    for _, row in false_positives.head(top_n).iterrows():
        print(f"  [{row.text_len} chars] {row.comment_text[:120]!r}")

    print("\n" + "="*60)
    print(f"ERROR ANALYSIS: {len(false_negatives)} False Negatives (Toxic → missed)")
    print("="*60)
    for _, row in false_negatives.head(top_n).iterrows():
        print(f"  [{row.text_len} chars] {row.comment_text[:120]!r}")

    # Check if error rate varies by text length - might reveal model weakness on short/long text
    print("\n" + "="*60)
    print("VULNERABILITY PATTERN: Text Length vs Error Rate")
    print("="*60)
    test_df["len_bucket"] = pd.cut(test_df.text_len, bins=[0,50,150,300,500,99999],
                                    labels=["<50","50-150","150-300","300-500","500+"])
    err_by_len = test_df.groupby("len_bucket").apply(
        lambda g: pd.Series({
            "total": len(g),
            "error_rate": ((g.pred != g.label).sum() / len(g) * 100).round(1)
        })
    )
    print(err_by_len)

    return false_positives, false_negatives

fp_df, fn_df = evaluate_and_analyze(trainer, test_df, test_dataset)

# Part 5: LLM Secondary Review (Optional)
# Use Claude to re-check false negatives - simulates human-in-the-loop moderation
import anthropic

def llm_content_review(texts, client):
    # Send texts to Claude for harm assessment - useful for high-risk cases model might miss
    results = []
    for text in texts:
        prompt = f"""You are a content safety reviewer focused on protecting minors.

Analyze the following text and determine if it is harmful to minors.

Text: "{text[:300]}"

Respond in this exact JSON format:
{{
  "is_harmful": true or false,
  "severity": "low" | "medium" | "high",
  "reason": "one sentence explanation",
  "category": "hate_speech" | "harassment" | "explicit" | "grooming" | "safe"
}}

Return only the JSON, no other text."""

        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=200,
            messages=[{"role": "user", "content": prompt}]
        )
        import json
        try:
            result = json.loads(response.content[0].text)
        except Exception:
            result = {"is_harmful": None, "reason": "parse error", "category": "unknown"}
        result["text"] = text[:100]
        results.append(result)
    return results

# Uncomment below and add your API key to run LLM review on missed cases
# client = anthropic.Anthropic(api_key="your_key_here")
# fn_samples = fn_df.comment_text.head(10).tolist()
# llm_results = llm_content_review(fn_samples, client)
# for r in llm_results:
#     print(r)

# Part 6: Save Model
model.save_pretrained("./minor_safety_model")
tokenizer.save_pretrained("./minor_safety_model")
print("Model saved to ./minor_safety_model")

# Part 7: Inference Example
# Single-text prediction: returns label (SAFE/HARMFUL) and confidence score
def predict(text, model, tokenizer, device="cuda"):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    prob = torch.softmax(logits, dim=1)[0][1].item()
    label = "HARMFUL" if prob > 0.5 else "SAFE"
    return {"label": label, "confidence": round(prob, 3)}

model.to("cuda" if torch.cuda.is_available() else "cpu")
device = "cuda" if torch.cuda.is_available() else "cpu"

# Test on a few examples to verify the model works as expected
examples = [
    "I love learning about science and nature!",
    "You are so stupid, nobody likes you",
    "Let's meet up after school, don't tell your parents",
]
print("\n--- Inference Examples ---")
for ex in examples:
    result = predict(ex, model, tokenizer, device)
    print(f"  {result['label']} ({result['confidence']:.1%}) | {ex[:60]!r}")

Dataset size: 20000 | Toxic: 10000 | Clean: 10000


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,0.175393,0.189604,0.928831,0.944034,0.914110
2,0.086448,0.200520,0.931980,0.925403,0.938650


...

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Training complete.



OVERALL PERFORMANCE
              precision    recall  f1-score   support

       Clean       0.94      0.93      0.93      2044
       Toxic       0.93      0.94      0.93      1956

    accuracy                           0.93      4000
   macro avg       0.93      0.93      0.93      4000
weighted avg       0.93      0.93      0.93      4000

Confusion Matrix:
[[1896  148]
 [ 120 1836]]
True Negatives: 1896 | False Positives: 148
False Negatives: 120 | True Positives: 1836

ERROR ANALYSIS: 148 False Positives (Clean → flagged as Toxic)
  [22 chars] 'Ya dude thats not cool'
  [141 chars] 'Hey, how are you? \n\nI remember back when you were just a Bavarian creampuff, long before you became a mass of twitchy mu'
  [64 chars] 'WISE IT LAD\n\nWAKA FLOCKABig textBig textBig textBig textBig text'
  [393 chars] 'Way to go!  Hope you get what you want!  But no matter what comes of it, deep down you know I am right about you, and yo'
  [214 chars] '"of birds form the planet Asia. Dogs were wo

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./minor_safety_model

--- Inference Examples ---
  SAFE (0.2%) | 'I love learning about science and nature!'
  HARMFUL (99.9%) | 'You are so stupid, nobody likes you'
  SAFE (0.7%) | "Let's meet up after school, don't tell your parents"
